# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.


In [ ]:
# List registered Record Sets
print("Available Record Sets (@id and name):")
record_set_ids = []
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets:
        print(f"- @id: {rs.id} | Name: {rs.name}")
        record_set_ids.append(rs.id)
else:
    # fallback if .record_sets is missing
    print("No record sets found via dataset.record_sets.")
    record_set_ids = []

# List fields in each record set
record_set_fields = {}
if record_set_ids:
    for rs in dataset.record_sets:
        print(f"\nFields for record set {rs.id}:")
        fields = getattr(rs, 'fields', [])
        record_set_fields[rs.id] = []
        for f in fields:
            print(f"  - @id: {f.id} | Name: {f.name} | DataType: {getattr(f, 'data_type', None)}")
            record_set_fields[rs.id].append({'id': f.id, 'name': f.name, 'data_type': getattr(f, 'data_type', None)})
else:
    print("Dataset has no record sets registered in metadata. Check `dataset.metadata` for details.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract all available record sets
dataframes = {}
if not record_set_ids:
    print("No record sets available to extract records from.")
else:
    for rs_id in record_set_ids:
        # Each record set is referenced by its @id
        print(f"Loading records for record set: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records, columns: {list(df.columns)}")
        except Exception as e:
            print(f"  Could not load records for {rs_id}: {e}")

    # Select the first record set for demo
    if len(record_set_ids) > 0:
        main_rs_id = record_set_ids[0]
        print(f"\nColumns in DataFrame for record set '{main_rs_id}':")
        print(dataframes[main_rs_id].columns.tolist())
        print("\nFirst five records:")
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers and grouping data by key attributes for further analysis.


In [ ]:
import numpy as np

# Example: Attempt basic EDA on the main record set (first record set found)
if record_set_ids:
    df = dataframes[main_rs_id]

    # Find a numeric field to analyze (example: look for 'coverage' or a field with float/int type)
    cand_numeric = None
    for field in record_set_fields.get(main_rs_id, []):
        # try to pick first float or int field (or one named coverage/mw/pI/peptide_count)
        name_lc = str(field['name']).lower()
        dt = field['data_type']
        if 'cover' in name_lc or 'mw' in name_lc or 'peptide' in name_lc or 'abundance' in name_lc or 'intensity' in name_lc or 'count' in name_lc or "pI" in name_lc:
            if cand_numeric is None and field['id'] in df.columns:
                # Test field type in the DF
                try:
                    vals = pd.to_numeric(df[field['id']], errors='coerce')
                    if not vals.isna().all():
                        cand_numeric = field['id']
                        break
                except Exception:
                    continue
        if dt and ('Float' in dt or 'Integer' in dt or 'Number' in dt):
            if cand_numeric is None and field['id'] in df.columns:
                # Try if the column parses as numeric
                vals = pd.to_numeric(df[field['id']], errors='coerce')
                if not vals.isna().all():
                    cand_numeric = field['id']
                    break
    # Fallback to first numeric column
    if cand_numeric is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            cand_numeric = numeric_cols[0]

    if cand_numeric is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using field '{cand_numeric}' for numeric EDA.")
        df_num = pd.to_numeric(df[cand_numeric], errors='coerce')
        threshold = df_num.mean()  # for demo, use mean as threshold
        filtered_df = df[df_num > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {cand_numeric} > {threshold:.2f}")

        # Normalization
        filtered_df[f"{cand_numeric}_normalized"] = (df_num - df_num.mean()) / df_num.std()
        print(f"Normalized '{cand_numeric}' for filtered records:")
        print(filtered_df[[cand_numeric, f"{cand_numeric}_normalized"].replace(' ', '_') if isinstance(cand_numeric, str) else cand_numeric].head())

        # Try grouping by a categorical field (e.g. 'description', 'sample', or any object dtype)
        group_field = None
        for field in record_set_fields.get(main_rs_id, []):
            if field['id'] in df.columns:
                if df[field['id']].dtype == 'object' and field['id'] != cand_numeric:
                    group_field = field['id']
                    break

        if group_field:
            print(f"Grouping by {group_field} and calculating mean of normalized {cand_numeric}:")
            grouped_df = filtered_df.groupby(group_field)[f"{cand_numeric}_normalized"].mean().to_frame()
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and cand_numeric is not None:
    plt.figure(figsize=(8, 5))
    # Histogram of the numeric column
    data = pd.to_numeric(df[cand_numeric], errors='coerce')
    sns.histplot(data.dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {cand_numeric}")
    plt.xlabel(cand_numeric)
    plt.ylabel('Frequency')
    plt.show()

    # If a group_field exists and is not too high cardinality, boxplot
    if 'group_field' in locals() and group_field and df[group_field].nunique() < 50:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field], y=data, showfliers=False)
        plt.xticks(rotation=90)
        plt.title(f"{cand_numeric} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(cand_numeric)
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
We have explored the FAIR⁲ dataset on mass spectrometry analysis of extracellular vesicles isolated from stimulated human mast cells using the `mlcroissant` library. We reviewed the record set structure using `@id`s, extracted records, performed basic filtering and normalization, and visualized distributions of selected numeric fields. For advanced applications, review the Croissant schema to select appropriate fields and record sets for domain-specific analysis.


<!-- End of notebook -->